### MSC_DA_CA2

## Sentiment Analysis using NLP (Natural Language Processing)

Dataset: EU Travel Survey on demand for innovative transport systems

https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/P82V9X
Description 	
The dataset comes from an EU-wide survey carried out in 2014 with the objective of gathering a number of transport and mobility indicators on transport user preferences with emphasis on the potential of emerging transport technologies and the acceptability of various transport policy measures. The CAWI (Computer Aided Web Interview) survey covered all 28 Member States of the European Union with the same questionnaire translated in the local languages. Samples of 1000 individuals in each country reflected the composition of adult population (from 16 years on) in terms of gender, age class, employment status, education level and living region. The report describing the survey can be accessed at: http://publications.jrc.ec.europa.eu/repository/handle/JRC96151

Keyword 	transport, mobility, survey, European Union

@data{DVN/P82V9X_2016,
author = {Christidis, Panayotis},
publisher = {Harvard Dataverse},
title = {{EU Travel Survey on demand for innovative transport systems}},
UNF = {UNF:6:dVj92hLsh1poq2//aVgVGA==},
year = {2016},
version = {V1},
doi = {10.7910/DVN/P82V9X},
url = {https://doi.org/10.7910/DVN/P82V9X}
}

In [269]:
#Importing the essential libraries, which are Pandas, Matplotlib, Numpy, and Seaborn.
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

# Set to display all columns
pd.set_option('display.max_columns', None)

When attempting to read a CSV file that is 'not encoded in UTF-8', which is the default encoding assumed by pandas.read_csv(), a UnicodeDecodeError was encountered. To read a file encoded differently, the alternative was to use the technique 'encoding='ISO-8859-1'.

In [270]:
eu_transp = pd.read_csv('eu_survey_transport_systems.csv', encoding='ISO-8859-1')
eu_transp.head(2)

,ID,Country,Gender,Age,Education,Region,Profession,Work_status,Household_members,Income_level,Location_of_resudence,Centre_or_suburbs,Public_transport_service,Car_driving_license,Number_vehicles_in_household,Considering_electric_or_hybrid_vehicle_next_purchase,Know_what_car_sharing_is,Would_subsribe_car_sharing_if_available,Most_frequent_trip_Walk,Most_frequent_trip_Bicycle,Most_frequent_trip_Car_as_Driver,Most_frequent_trip_Car_as_Passenger,Most_frequent_trip_Train,Most_frequent_trip_Underground_or_light_train,Most_frequent_trip_Tram,Most_frequent_trip_Bus,Most_frequent_trip_Motorcycle_or_moped,Destination_most_frequent_trip,Frequency_most_frequent_trip,Problem_most_frequent_trip_Congestion,Problem_most_frequent_trip_Parking,Problem_most_frequent_trip_Lack_of_bicycle_lanes,Problem_most_frequent_trip_Infrequent_public_transport,Problem_most_frequent_trip_Lack_of_public_transport_coverage,Problem_most_frequent_trip_none,Transfers_between_modes_during_frequent_trip,Frequent_trip_duration_in_minutes,Frequent_trip_distance,Concern_environmental_impacts,Preference_tolls_or_traffic_limitation
0,1,Belgium,Female,49,Upper secondary (high school or similar);,Prov. Oost-Vlaanderen,housewife,Not Employed,two,lower middle,Metropolitan area of a big city with more than...,in the suburbs,Well served by public transport,Yes,1,Maybe yes maybe not,Yes,"Maybe yes, maybe not. I would need to test the...",No,No,Yes,No,No,No,No,No,No,It is outside an urban area,Make this trip every day/ every working day of...,Yes,Yes,No,No,No,No,,20,3-5 KM,5,No preferences
1,2,France,Male,26,"Tertiary and higher (University degree, PhD or...",Pays de la Loire,unemployed,Not Employed,four,low,Small or medium town (less than 250.000 inhabi...,in the centre of the city,Difficult to reach with public transport,Yes,3,Probably not,Yes,"Maybe yes, maybe not. I would need to test the...",No,No,Yes,No,No,No,No,No,No,"In an urban area, different from where I live",Make this trip 2-3 days per week,No,Yes,No,Yes,Yes,No,,20,11-20 KM,8,Probably more acceptable to limit road traffic


In [271]:
eu_transp.shape

(26605, 40)

### Data Preprocessing / Text Preprocessing:

In [272]:
eu_transp.isnull().values.any()

False

In [273]:
# Print the unique country values
eu_transp = pd.DataFrame(eu_transp)
unique_countries = eu_transp['Country'].unique()
print(unique_countries)

['Belgium' 'France' 'Czech Republic' 'Sweden' 'Poland' 'Hungary' 'Germany'
 'Spain' 'Denmark' 'Italy' 'Finland' 'Netherlands' 'Romania' 'Portugal'
 'Lithuania' 'Ireland' 'Greece' 'Great Britain' 'Estonia' 'Slovenia'
 'Latvia' 'Croatia' 'Austria' 'Bulgaria' 'Slovakia' 'Malta' 'Louxembourg'
 'Cyprus']


#### Filter the dataset countries & define the columns for sentiment analysis

In [274]:
# List of Specific Countries
countries = ['Ireland', 'Denmark', 'Belgium', 'Sweden', 'Finland', 'Portugal']
filtered_df = eu_transp[eu_transp['Country'].isin(countries)]

# Select Relevant Columns for Sentiment Analysis and lowercasing the text
for col in ['Considering_electric_or_hybrid_vehicle_next_purchase', 
            'Would_subsribe_car_sharing_if_available', 
            'Preference_tolls_or_traffic_limitation']:
    filtered_df[col] = filtered_df[col].str.lower()

# Define the columns for sentiment analysis
sentiment_columns = ['Considering_electric_or_hybrid_vehicle_next_purchase', 
                     'Would_subsribe_car_sharing_if_available', 
                     'Preference_tolls_or_traffic_limitation']

# Keep the 'Country' column along with the sentiment columns
sentiment_df = filtered_df[['Country'] + sentiment_columns]
sentiment_df.head(2)

,Country,Considering_electric_or_hybrid_vehicle_next_purchase,Would_subsribe_car_sharing_if_available,Preference_tolls_or_traffic_limitation
0,Belgium,maybe yes maybe not,"maybe yes, maybe not. i would need to test the...",no preferences
3,Sweden,maybe yes maybe not,don't know / no answer,probably more acceptable to pay for less conge...


Converting text in specific columns to lowercase: This standardises the text data and makes it more homogeneous for analysis, which is especially crucial for textual and sentiment analysis.

Defining Relevant Columns for Analysis: Creating a new DataFrame sentiment_df with only the columns needed for sentiment analysis, enabling a focused and efficient analysis process.

In [275]:
sentiment_df.shape

(6033, 4)

In [276]:
# Print the unique country values
sentiment_df = pd.DataFrame(sentiment_df)
unique_countries = sentiment_df['Country'].unique()
print(unique_countries)

['Belgium' 'Sweden' 'Denmark' 'Finland' 'Portugal' 'Ireland']


## Aggregate and Analyze Sentiments with TextBlob:

Data Aggregation:  Summarising and presenting data for interpretation.

Create a Sentiment Analysis Function: Using TextBlob , define a sentiment analysis function.

Apply the following function on text columns: For each sentiment analysis result, add a new column to dataset.

In [277]:
# Define the sentiment analysis function
from textblob import TextBlob

def get_sentiment(text):
    if pd.isna(text):
        return None
    return TextBlob(text).sentiment.polarity

# List of columns to analyze
columns_to_analyze = [
    'Considering_electric_or_hybrid_vehicle_next_purchase', 
    'Would_subsribe_car_sharing_if_available', 
    'Preference_tolls_or_traffic_limitation'
]
# Apply sentiment analysis to each relevant column
for col in columns_to_analyze:
    sentiment_col_name = col + '_sentiment'
    sentiment_df[sentiment_col_name] = sentiment_df[col].apply(get_sentiment)

# Display the first few rows of the DataFrame with new sentiment columns
sentiment_df.head(2)

,Country,Considering_electric_or_hybrid_vehicle_next_purchase,Would_subsribe_car_sharing_if_available,Preference_tolls_or_traffic_limitation,Considering_electric_or_hybrid_vehicle_next_purchase_sentiment,Would_subsribe_car_sharing_if_available_sentiment,Preference_tolls_or_traffic_limitation_sentiment
0,Belgium,maybe yes maybe not,"maybe yes, maybe not. i would need to test the...",no preferences,0.0,0.0,0.000000
3,Sweden,maybe yes maybe not,don't know / no answer,probably more acceptable to pay for less conge...,0.0,0.0,0.166667


Creating New Columns with Sentiment Scores: Creating new columns in the DataFrame to indicate the emotion score of the associated text column. This step is critical for transforming qualitative text data into quantitative sentiment scores.

Grouping and Aggregating Data: This stage is critical for summarising the data and allowing easy comparison of sentiment across countries.

#### Analyze and Compare Sentiments
This could involve grouping by country, taking averages, plotting results.

This aggregation will display the average sentiment in each selected country for each component of transportation (electric/hybrid vehicle consideration, car sharing subscription, and tolls/traffic constraints). The analysis should then be refined to meet specific research objectives.

In [278]:
# Aggregate the average sentiment by country
average_sentiments = sentiment_df.groupby('Country').agg({
    'Considering_electric_or_hybrid_vehicle_next_purchase_sentiment': 'mean',
    'Would_subsribe_car_sharing_if_available_sentiment': 'mean',
    'Preference_tolls_or_traffic_limitation_sentiment': 'mean'
})

# Display the average sentiments by country
print(average_sentiments)

          Considering_electric_or_hybrid_vehicle_next_purchase_sentiment  \
Country                                                                    
Belgium                                           -0.006000                
Denmark                                           -0.010184                
Finland                                            0.002452                
Ireland                                            0.018857                
Portugal                                           0.026099                
Sweden                                             0.011205                

          Would_subsribe_car_sharing_if_available_sentiment  \
Country                                                       
Belgium                                            0.125795   
Denmark                                            0.153704   
Finland                                            0.112759   
Ireland                                            0.118905   
Portugal     

The sentiment analysis results represent the average sentiment scores for each of the three transportation aspects (electric/hybrid vehicle consideration, car sharing subscription, and tolls/traffic limitation preferences) in each of the selected countries (Belgium, Denmark, Finland, Ireland, Portugal, and Sweden).

- Sentiment Scores: Sentiment scores normally vary between -1.0 and 1.0. A score around -1.0 indicates a strongly negative sentiment, a score near 1.0 shows a strongly positive feeling, and a score near 0 suggests a neutral sentiment or a lack of distinct positive/negative language.

- Considering Electric or Hybrid Vehicle Next Purchase Sentiment: Belgium and Denmark have slightly negative sentiment (-0.006 and -0.010). Finland, Ireland, Portugal, and Sweden all had slightly favourable sentiment, with Portugal having the greatest (0.026).

Insight: There appears to be a little preference for electric or hybrid vehicles in Ireland, Portugal, and Sweden, whilst mood is slightly unfavourable in Belgium and Denmark.

- Would Subscribe Car Sharing if Available Sentiment: These scores show attitudes towards using automobile sharing services if they are available. Belgium, Denmark, Finland, Ireland, and Sweden all have positive sentiment, with Denmark having the highest (0.154). Portugal, positive sentiment is lower (0.055).

Insight: Car-sharing services are usually regarded well in these nations, particularly in Denmark and Sweden, indicating a potential market for these services.

- Preference Tolls or Traffic Limitation Sentiment: These scores represent people's feelings towards tolls or traffic restrictions. Positive results suggest a favourable attitude towards these measures. Positive sentiment in all countries, with Finland having the highest positivity (0.281).

Insight: Across all of these countries, there is a widespread preference for tolls or traffic restriction measures, presumably indicating a willingness to support measures that reduce traffic congestion and possibly environmental damage.

Conclusion: The sentiment scores provide insight into the general sentiments or ideas held by people in these nations about various transportation-related topics.
A positive score shows a proclivity for positive opinions, a negative score suggests a proclivity for negative opinions, and scores near zero indicate a proclivity for neutral thoughts or a combination of positive and negative attitudes.
These findings can be used to assess public perception or adoption of various transport policies or technologies across countries. However, it is crucial to note that the manner in which emotions are represented can vary substantially depending on cultural and language situations. These ratings should be viewed with these potential variances in mind.

### Machine Learning Classification 
Let's us use a machine learning model to perform a categorization problem.

Classifying the sentiment of the responses into categories such as "Positive," "Negative," and "Neutral." This will entail converting textual input to numerical format and then training a classifier.

# Additional Notes:
Labeling Data: If your dataset doesn't already have sentiment labels, you'll need to label your data manually or use a pre-trained sentiment analysis model to generate these labels.
Choosing a Model: While Naive Bayes is a good starting point, consider experimenting with different models (like Logistic Regression, SVM, or even neural networks) depending on your dataset and accuracy requirements.
Parameter Tuning: Experiment with different parameters in CountVectorizer and your model to optimize performance.
This guide provides a basic framework for text classification. You may need to adjust the steps based on the specifics of your dataset and analysis goals.

In [279]:
# Importing the required libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.metrics import classification_report, accuracy_score
import string
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rosil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rosil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Text Preprocessing

#### Combine and Clean Text Dataset
Let's analyze the sentiment of combined text from multiple columns

In [280]:
# List of Specific Countries
countries = ['Ireland', 'Denmark', 'Belgium', 'Norway', 'Finland', 'Portugal']
transp_sentiment_df = eu_transp[eu_transp['Country'].isin(countries)]

# Combine and clean the text columns:
transp_sentiment_df['combined_text'] = transp_sentiment_df[['Considering_electric_or_hybrid_vehicle_next_purchase', 
                                                            'Would_subsribe_car_sharing_if_available', 
                                                            'Preference_tolls_or_traffic_limitation']].agg(' '.join, axis=1)
transp_sentiment_df['combined_text'] = transp_sentiment_df['combined_text'].str.lower()

# Select only the columns you need, such as 'Country' and 'combined_text':
transp_systems_df = transp_sentiment_df[['Country', 'combined_text']]
transp_systems_df.head()

,Country,combined_text
0,Belgium,"maybe yes maybe not maybe yes, maybe not. i wo..."
5,Belgium,don't know/no answer don't know / no answer no...
9,Belgium,"maybe yes maybe not no, i would not be interes..."
13,Denmark,"certainly not no, i would not be interested in..."
15,Finland,"probably yes yes, instead of purchasing a new ..."


In [281]:
transp_systems_df.shape

(5029, 2)

#### Label Creation
Purpose: It focuses on labelling the text's sentiment. Use the TextBlob library to analyse the sentiment polarity of each text input in this scenario.

In [282]:
# Using TextBlob for automatic labeling
def label_sentiment(text):
    sentiment = TextBlob(text).sentiment.polarity
    if sentiment > 0:
        return 'Positive'
    elif sentiment < 0:
        return 'Negative'
    else:
        return 'Neutral'

transp_systems_df['sentiment_label'] = transp_systems_df['combined_text'].apply(label_sentiment)
transp_systems_df.head(2)

,Country,combined_text,sentiment_label
0,Belgium,"maybe yes maybe not maybe yes, maybe not. i wo...",Neutral
5,Belgium,don't know/no answer don't know / no answer no...,Neutral


Sentiment Labeling: To begin, use TextBlob to analyse and label each text entry in your DataFrame. Based on the attitude expressed in the text, this stage assigns each text to one of three groups (Positive, Negative, or Neutral).

## Multinomial Naive Bayes Classifier:
Let's start with NB Classifier because it's a good starting point for text classification, especially with a moderate-sized. It is simple, fast, and frequently unexpectedly effective. And also suitable for datasets where the features are independent of each other.

#### Feature Extraction
Transform text data into a numerical format suitable for machine learning (e.g., using CountVectorizer or TfidfVectorizer).
Create Bag of Words

# Convert the text data into a numerical format using CountVectorizer:
vectorizer = CountVectorizer(stop_words=stopwords.words('english'), max_features=1000)
X = vectorizer.fit_transform(transp_systems_df['combined_text'])

#For demonstration, let's 
y = transp_systems_df['sentiment_label']  

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#### Train a Machine Learning Model

model = MultinomialNB()
model.fit(X_train, y_train)

#### Model Evaluation
Make Predictions and Evaluate the Model

#Evaluate the model on the test set
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

- Accuracy (0.9821 or 98.21%): The percentage of total right predictions (both positive and negative) made out of all forecasts made. An accuracy of 98.21% implies that the model accurately identified the sentiment of the text on your test dataset 98.21% of the time.
- The classification report includes additional specific metrics for each class (Negative, Neutral, and Positive):

The precision is defined as the proportion of accurately predicted positive observations to total expected positives. High precision is associated with a low false-positive rate.
- For 'Negative,' 0.48 (48%) indicates that when the model predicts a negative event, it is true 48% of the time.
-For 'Neutral' and 'Positive,' 1.00 (100%) indicates that the model predicts these classifications quite accurately.

Recall (Sensitivity): The proportion of accurately predicted positive observations to the total number of observations in the class.
- For 'Negative,' a value of 1.00 (100%) indicates that the model correctly detected all genuine negative cases.
- The value 0.98 (98%) for 'Positive' shows that the model correctly detected 98% of all actual positive events.
F1-Score: The weighted average of Precision and Recall is the F1-score. It considers both false positives and false negatives. It is a fantastic technique to demonstrate that a class has a good mix of precision and recall.
- For 'Negative': 0.65 (65%) shows a moderate balance between precision and recall for this class.
- Scores for 'Neutral' and 'Positive' are near to 1.00 (100%), suggesting good precision and recall balance.

Support: The number of instances of the specified class in the supplied dataset. In your test set, for example, there are 24 instances of the 'Negative' class.

Macro Average and Weighted Average:
- Macro Avg: The mean precision, recall, and F1-score across all classes. The macro average does not account for label imbalance. The macro average is high in this case (0.83 for precision, 0.99 for recall, and 0.88 for F1-score), but it is skewed by the excellent performance in the 'Neutral' and 'Positive' classes.
- Weighted Avg: The weighted average of precision, recall, and F1-score between classes. When there is a class imbalance, such as in your dataset, this is more useful. The weighted averages are quite high (0.99 for precision, 0.98 for recall, and 0.99 for F1-score), demonstrating good performance across all classes, with a preference for the more frequent classes.

Conclusion: The model does an excellent job of distinguishing 'Neutral' and 'Positive' moods.
However, the model appears to struggle with precision in the 'Negative' class (48%) but has perfect recall, implying that it labels too many instances as negative (high false positives) but does not miss the true negative occurrences.

The high accuracy indicates generally good performance, but the lesser precision for 'Negative' indicates that the model may be over-predicting this class. This could be an area for further research and development, especially in datasets with imbalanced classes.

### Text Preprocessing

#### Bag of Words ( BoW) using CountVectorizer

Let's go further to preprocess the text data even more (tokenization, stopword elimination, and stemming). This process focuses on cleaning and standardising the text data.

In [283]:
def preprocess_text(text):
    # Tokenization
    tokens = word_tokenize(text)
    # Removing stopwords
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    # Stemming
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]
    return ' '.join(stemmed_tokens)

# Apply preprocessing to the 'combined_text' column
transp_systems_df['processed_text'] = transp_systems_df['combined_text'].apply(preprocess_text)
transp_systems_df.head(2)

,Country,combined_text,sentiment_label,processed_text
0,Belgium,"maybe yes maybe not maybe yes, maybe not. i wo...",Neutral,"mayb ye mayb mayb ye , mayb . would need test ..."
5,Belgium,don't know/no answer don't know / no answer no...,Neutral,n't know/no answer n't know / answer prefer


Additional text pretreatment techniques, such as tokenization, stopword removal, and stemming, are critical in preparing data for sentiment analysis models. Textual data fine-tuning is a regular method in text-based machine learning projects, with the goal of improving model performance by decreasing noise while maintaining relevant information.

### Multinomial Naive Bayes

#### Split the dataset into training and testing parts

In [284]:
X = transp_systems_df['processed_text']
y = transp_systems_df['sentiment_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#### Apply CountVectorizer to X_train and X_test

In [285]:
cv = CountVectorizer(max_features=3000, ngram_range=(1,1), stop_words='english')
X_train_counts = cv.fit_transform(X_train)
X_test_counts = cv.transform(X_test)

#### Model Training and Evaluation for Multinomial Naive Bayes:

In [286]:
mnb_model = MultinomialNB()
mnb_model.fit(X_train_counts, y_train)
y_pred_mnb = mnb_model.predict(X_test_counts)

print("Multinomial Naive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_mnb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_mnb))

Multinomial Naive Bayes
Accuracy: 0.9098740888005301

Classification Report:
               precision    recall  f1-score   support

    Negative       0.57      1.00      0.73        24
     Neutral       0.70      1.00      0.82       272
    Positive       1.00      0.89      0.94      1213

    accuracy                           0.91      1509
   macro avg       0.76      0.96      0.83      1509
weighted avg       0.94      0.91      0.92      1509



##### The Multinomial Naive Bayes (MNB) model's performance on the test dataset. 
Accuracy: 0.9098740888005301 (or about 91%): This means that roughly 91% of the model's predictions are right. In other words, the model properly labels the text's sentiment (91% of the time) as Negative, Neutral, or Positive.

Classification Report:
- Negative Class: Precision: 0.57 - The model predicts 'Negative' 57% of the time.
Recall: 1.00 - The model accurately detects all 'Negative' occurrences. F1-Score: 0.73 - The F1-Score is a harmonic mean of precision and recall that indicates a balance. A higher F1-score is preferable, while 0.73 is adequate.
- Neutral Class: Precision: 0.70 - The model correctly predicts 'Neutral' 70% of the time.
Recall: 1.00 - The model correctly detects 100% of the 'Neutral' situations. F1-score: 0.82 - A good F1-score for the 'Neutral' class, showing a fair balance of precision and recall.
- Positive Class: Precision: 1.00 - The model's 'Positive' predictions are exceptionally accurate.
The model properly detects 89% of the real 'Positive' cases with a recall of 0.89. F1-score: 0.94 - This is an extremely high F1-score, indicating great performance in the 'Positive' class.

- Macro Average: This determines the average metrics across classes, giving each class equal weight. The macro average does well here (F1-score of 0.83), but it does not adjust for class imbalance.
- Weighted Average: This compensates for class imbalance by weighting the metrics based on the number of true instances in each class. The average weighted F1-score is 0.92, suggesting excellent overall performance.

Insights and Interpretation
- Strong Positive Detection: The model is especially good at detecting 'Positive' feelings.
High Recall for Negative and Neutral: The model has a high recall for the 'Negative' and 'Neutral' classes, indicating that it is effective at detecting these cases but at the expense of lower precision (more false positives).

- Overall Good Performance: The model performs well on this dataset, as evidenced by its 91% accuracy and strong F1-scores for each class.
Negative Class Overfitting: The 100% recall and lesser precision for 'Negative' and 'Neutral' may indicate overfitting or an imbalance in the dataset.

In this dataset, the Multinomial Naive Bayes model performs well in sentiment classification. However, the precision-recall trade-offs for each class, as well as the possibility of overfitting or imbalance for specific classes, may be topics to investigate for model improvement.


### Bernoulli Naive Bayes:

In [287]:
binary_vectorizer = CountVectorizer(binary=True)
X_train_binary = binary_vectorizer.fit_transform(X_train)
X_test_binary = binary_vectorizer.transform(X_test)

#### Model Training and Evaluation for Multinomial Naive Bayes:

In [288]:
bnb_model = BernoulliNB()
bnb_model.fit(X_train_binary, y_train)
y_pred_bnb = bnb_model.predict(X_test_binary)

print("Bernoulli Naive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_bnb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_bnb))

Bernoulli Naive Bayes
Accuracy: 0.9821073558648111

Classification Report:
               precision    recall  f1-score   support

    Negative       0.48      1.00      0.65        24
     Neutral       1.00      1.00      1.00       272
    Positive       1.00      0.98      0.99      1213

    accuracy                           0.98      1509
   macro avg       0.83      0.99      0.88      1509
weighted avg       0.99      0.98      0.99      1509



### Feature Generation using TF-IDF